# NAOPC Benchmark

Ноутбук для сравнения `IG`, `Cheap-IG` и `NAA` по метрике `NAOPC` на `100` изображениях из `Oxford Pets`.

По умолчанию benchmark работает в режиме `unit_mode="spatial_cell"`, то есть perturbation-юнитом считается пространственная клетка слоя с агрегацией по каналам. Режим `filter` сохранён как опция.

Нормализация `NAOPC` по умолчанию считается в режиме `limit_mode="beam"` с `beam_size=5`, то есть `lower/upper` limits ищутся beam-search по алгоритму из оригинальной статьи. Режим `limit_mode="exact"` оставлен для маленьких candidate-наборов.

## Импорты

In [8]:
from pathlib import Path

from IPython.display import Markdown, display

from modules.naopc_benchmark import benchmark_classifier_naopc, classifier_method_spec


## Параметры

Первый запуск на uncached `100` изображениях может быть долгим. Повторные запуски должны опираться на `core-cache` и `method-cache`.

In [9]:
OXFORD_PETS_DIR = Path("oxford_pets")
N_IMAGES = 100

CLASSIFIER_LAYER = "model.6"
UNIT_MODE = "spatial_cell"  # alternative: "filter"
N_STEPS = 128
CANDIDATE_TOP_K = 16000
LIMIT_MODE = "beam"  # alternative: "exact"
BEAM_SIZE = 5
CLEAR_EVERY = 8
FD_EPS = 1e-3

CHEAP_IG_SEGMENT_START = 0.0
CHEAP_IG_SEGMENT_END = 0.1
CHEAP_IG_SELECTION_MODE = "signed"
CHEAP_IG_SELECTION_TOP_K = 5000

CACHE_ROOT = Path("output/naopc_cache")
OUTPUT_DIR = Path(f"output/naopc_classifier_{UNIT_MODE}_{LIMIT_MODE}_beam_{BEAM_SIZE}_oxford_pets_100")
REFRESH_CORE = False
REFRESH_METHODS = False


In [10]:
def collect_oxford_pets_images(image_dir=OXFORD_PETS_DIR, n_images=N_IMAGES):
    image_dir = Path(image_dir)
    if not image_dir.exists():
        raise FileNotFoundError(f"Oxford Pets directory not found: {image_dir}")

    image_paths = []
    for pattern in ("*.jpg", "*.jpeg", "*.png", "*.webp"):
        image_paths.extend(image_dir.glob(pattern))

    image_paths = sorted(set(image_paths), key=lambda path: path.name.lower())
    if len(image_paths) < n_images:
        raise ValueError(
            f"Requested {n_images} images, but found only {len(image_paths)} in {image_dir}"
        )
    return [str(path) for path in image_paths[:n_images]]


IMAGE_PATHS = collect_oxford_pets_images()
len(IMAGE_PATHS), IMAGE_PATHS[:5]


(100,
 ['oxford_pets/Abyssinian_1.jpg',
  'oxford_pets/Abyssinian_108.jpg',
  'oxford_pets/Abyssinian_117.jpg',
  'oxford_pets/Abyssinian_126.jpg',
  'oxford_pets/Abyssinian_135.jpg'])

## Методы

In [11]:
METHOD_SPECS = [
    classifier_method_spec("ig", name="IG"),
    classifier_method_spec(
        "cheap_ig",
        name="Cheap-IG",
        segment_start=CHEAP_IG_SEGMENT_START,
        segment_end=CHEAP_IG_SEGMENT_END,
        selection_mode=CHEAP_IG_SELECTION_MODE,
        selection_top_k=CHEAP_IG_SELECTION_TOP_K,
    ),
    classifier_method_spec("naa", name="NAA"),
]

METHOD_SPECS


[{'kind': 'ig', 'name': 'IG'},
 {'kind': 'cheap_ig',
  'segment_start': 0.0,
  'segment_end': 0.1,
  'selection_mode': 'signed',
  'selection_top_k': 5000,
  'name': 'Cheap-IG'},
 {'kind': 'naa', 'name': 'NAA'}]

## Запуск Бенчмарка

In [12]:
results = benchmark_classifier_naopc(
    image_paths=IMAGE_PATHS,
    method_specs=METHOD_SPECS,
    layer_name=CLASSIFIER_LAYER,
    unit_mode=UNIT_MODE,
    n_steps=N_STEPS,
    candidate_top_k=CANDIDATE_TOP_K,
    limit_mode=LIMIT_MODE,
    beam_size=BEAM_SIZE,
    clear_every=CLEAR_EVERY,
    fd_eps=FD_EPS,
    cache_root=CACHE_ROOT,
    target_dir=OUTPUT_DIR,
    save_output=True,
    refresh_core=REFRESH_CORE,
    refresh_methods=REFRESH_METHODS,
)

print("output_dir:", results["output_dir"])
print("report_path:", results["report_path"])
print("summary_path:", results["summary_path"])


KeyboardInterrupt: 

## Markdown-Отчёт

In [ ]:
display(Markdown(results["report_markdown"]))


# NAOPC Benchmark

- task=`classifier`
- layer_name=`model.6`
- n_steps=128
- unit_mode=`spatial_cell`
- candidate_top_k=10
- limit_mode=`beam`
- beam_size=5
- n_images=100
- cache_root=`output/naopc_cache`
- clear_every=8
- fd_eps=0.001
- top_n=0

## Aggregate Summary

| Method | NAOPC | AOPC | Mean Rank | Attr Runtime (s) | Eval Runtime (s) | Abs Error |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| IG | 0.7287 +- 0.1961 | 0.8239 +- 1.5068 | 2.0900 +- 0.6016 | 4.8112 +- 0.2012 | 0.4374 +- 0.0375 | 0.7237 +- 0.6076 |
| Cheap-IG | 0.6611 +- 0.2221 | 0.6710 +- 1.5231 | 2.4900 +- 0.7141 | 2.4239 +- 0.1300 | 0.4347 +- 0.0322 | 101.8435 +- 37.4375 |
| NAA | 0.8790 +- 0.1073 | 1.0766 +- 1.2829 | 1.4200 +- 0.7373 | 2.1094 +- 0.1038 | 0.4354 +- 0.0322 | 13.1379 +- 3.1544 |

## Core Limits

| Metric | Value |
| --- | ---: |
| upper_limit | 1.3243 +- 1.3494 |
| lower_limit | -0.8226 +- 0.8755 |
| clean_delta | 14.0641 +- 2.9601 |
| subset_eval_count | 223.8500 +- 15.6680 |
| core_runtime_s | 1.4436 +- 0.1333 |

## Figures

### naopc_summary

![](output/naopc_classifier_spatial_cell_beam_beam_5_oxford_pets_100/figures/naopc_summary.png)

### naopc_distributions

![](output/naopc_classifier_spatial_cell_beam_beam_5_oxford_pets_100/figures/naopc_distributions.png)

### naopc_curves

![](output/naopc_classifier_spatial_cell_beam_beam_5_oxford_pets_100/figures/naopc_curves.png)

### naopc_pairwise_wins

![](output/naopc_classifier_spatial_cell_beam_beam_5_oxford_pets_100/figures/naopc_pairwise_wins.png)

## Per-Image NAOPC

| Image | IG | Cheap-IG | NAA |
| --- | ---: | ---: | ---: |
| Abyssinian_1.jpg | 0.9462 | 0.8801 | 0.9253 |
| Abyssinian_108.jpg | 0.5660 | 0.4583 | 0.9702 |
| Abyssinian_117.jpg | 0.9309 | 0.7378 | 0.9836 |
| Abyssinian_126.jpg | 0.7779 | 0.8464 | 0.8540 |
| Abyssinian_135.jpg | 0.6130 | 0.6142 | 0.8915 |
| Abyssinian_144.jpg | 0.7221 | 0.6984 | 0.8767 |
| Abyssinian_154.jpg | 0.7095 | 0.6700 | 0.6026 |
| Abyssinian_165.jpg | 0.9095 | 0.9216 | 0.9050 |
| Abyssinian_175.jpg | 0.6491 | 0.8687 | 0.9323 |
| Abyssinian_184.jpg | 0.9442 | 0.9920 | 0.9779 |
| Abyssinian_2.jpg | 0.7506 | 0.7421 | 0.8910 |
| Abyssinian_212.jpg | 0.9052 | 0.8804 | 0.7883 |
| Abyssinian_224.jpg | 0.6857 | 0.7260 | 0.8863 |
| Abyssinian_29.jpg | 0.7201 | 0.7705 | 0.6984 |
| Abyssinian_43.jpg | 0.9373 | 0.8652 | 0.7592 |
| Abyssinian_52.jpg | 0.6668 | 0.6369 | 0.9606 |
| Abyssinian_63.jpg | 0.6711 | 0.4976 | 0.9574 |
| Abyssinian_73.jpg | 0.6267 | 0.6668 | 0.8039 |
| Abyssinian_83.jpg | 0.6742 | 0.6667 | 0.9021 |
| Abyssinian_92.jpg | 0.6576 | 0.5424 | 0.9931 |
| american_bulldog_108.jpg | 0.5876 | 0.6563 | 0.8069 |
| american_bulldog_117.jpg | 0.6915 | 0.6166 | 0.8340 |
| american_bulldog_126.jpg | 0.8096 | 0.8250 | 0.9268 |
| american_bulldog_135.jpg | 0.9815 | 0.8861 | 0.8938 |
| american_bulldog_144.jpg | 0.5153 | 0.3980 | 0.9435 |
| american_bulldog_158.jpg | 0.7706 | 0.7386 | 0.8821 |
| american_bulldog_173.jpg | 0.6083 | 0.6995 | 0.8768 |
| american_bulldog_182.jpg | 0.6383 | 0.7303 | 0.9640 |
| american_bulldog_191.jpg | 0.7381 | 0.7308 | 0.9335 |
| american_bulldog_200.jpg | 0.7968 | 0.4166 | 0.8504 |
| american_bulldog_212.jpg | 0.9041 | 0.8135 | 0.9485 |
| american_bulldog_24.jpg | 0.8089 | 0.7232 | 0.8316 |
| american_bulldog_33.jpg | 0.8718 | 0.7219 | 0.8845 |
| american_bulldog_43.jpg | 0.3451 | 0.1686 | 0.9402 |
| american_bulldog_52.jpg | 0.5697 | 0.3857 | 0.8278 |
| american_bulldog_61.jpg | 0.8924 | 0.9008 | 0.8755 |
| american_bulldog_70.jpg | 0.6431 | 0.5103 | 0.9723 |
| american_bulldog_8.jpg | 0.4121 | 0.5986 | 0.9570 |
| american_bulldog_9.jpg | 0.5736 | 0.5644 | 0.9731 |
| american_bulldog_99.jpg | 0.7600 | 0.7508 | 0.8145 |
| american_pit_bull_terrier_107.jpg | 0.7961 | 0.7886 | 0.9023 |
| american_pit_bull_terrier_116.jpg | 0.7577 | 0.2187 | 0.9789 |
| american_pit_bull_terrier_125.jpg | 0.7328 | 0.4965 | 0.7270 |
| american_pit_bull_terrier_134.jpg | 0.8159 | 0.8125 | 0.9134 |
| american_pit_bull_terrier_143.jpg | 0.9920 | 0.9920 | 0.9156 |
| american_pit_bull_terrier_152.jpg | 0.3883 | 0.4246 | 0.7745 |
| american_pit_bull_terrier_161.jpg | 0.7924 | 0.8168 | 0.8720 |
| american_pit_bull_terrier_170.jpg | 0.8377 | 0.6589 | 0.9233 |
| american_pit_bull_terrier_18.jpg | 0.8621 | 0.8056 | 0.9948 |
| american_pit_bull_terrier_189.jpg | 0.9220 | 0.9433 | 0.9686 |
| american_pit_bull_terrier_198.jpg | 0.6616 | 0.5577 | 0.9201 |
| american_pit_bull_terrier_22.jpg | 0.9626 | 0.9729 | 0.9748 |
| american_pit_bull_terrier_32.jpg | 0.7026 | 0.6091 | 0.9337 |
| american_pit_bull_terrier_42.jpg | 0.1457 | 0.1739 | 0.4383 |
| american_pit_bull_terrier_51.jpg | 0.7920 | 0.4699 | 0.9497 |
| american_pit_bull_terrier_60.jpg | 0.8324 | 0.7105 | 0.9061 |
| american_pit_bull_terrier_7.jpg | 0.9273 | 0.8139 | 0.9759 |
| american_pit_bull_terrier_79.jpg | 0.6988 | 0.5068 | 0.9706 |
| american_pit_bull_terrier_9.jpg | 0.7370 | 0.5338 | 0.8921 |
| american_pit_bull_terrier_99.jpg | 0.5461 | 0.5606 | 0.8364 |
| basset_hound_107.jpg | 0.8919 | 0.7102 | 0.9321 |
| basset_hound_116.jpg | 0.8992 | 0.6842 | 0.9762 |
| basset_hound_125.jpg | 0.1672 | 0.0343 | 0.9464 |
| basset_hound_134.jpg | 0.8930 | 0.8672 | 0.9783 |
| basset_hound_143.jpg | 0.8288 | 0.6931 | 0.8083 |
| basset_hound_152.jpg | 0.8871 | 0.9398 | 0.9090 |
| basset_hound_161.jpg | 0.7188 | 0.7363 | 0.6756 |
| basset_hound_170.jpg | 0.4785 | 0.3614 | 0.9809 |
| basset_hound_18.jpg | 0.3986 | 0.3049 | 0.7575 |
| basset_hound_189.jpg | 0.5401 | 0.4755 | 0.9923 |
| basset_hound_198.jpg | 0.6940 | 0.3625 | 0.8940 |
| basset_hound_26.jpg | 0.9259 | 0.8614 | 0.8301 |
| basset_hound_35.jpg | 0.9583 | 0.9923 | 0.9584 |
| basset_hound_44.jpg | 0.9638 | 0.6887 | 0.9466 |
| basset_hound_53.jpg | 0.1104 | 0.1003 | 0.8735 |
| basset_hound_62.jpg | 0.9622 | 0.9845 | 0.7672 |
| basset_hound_71.jpg | 0.8105 | 0.7002 | 0.7585 |
| basset_hound_80.jpg | 0.9129 | 0.9403 | 0.8553 |
| basset_hound_9.jpg | 0.6682 | 0.4883 | 0.8216 |
| basset_hound_99.jpg | 0.7801 | 0.5181 | 0.9885 |
| beagle_108.jpg | 0.8150 | 0.7191 | 0.7967 |
| beagle_118.jpg | 0.7258 | 0.8523 | 0.4521 |
| beagle_127.jpg | 0.8216 | 0.8770 | 0.8585 |
| beagle_137.jpg | 0.9634 | 0.8546 | 0.9942 |
| beagle_146.jpg | 0.8915 | 0.3728 | 0.8136 |
| beagle_155.jpg | 0.7070 | 0.5074 | 0.8394 |
| beagle_165.jpg | 0.2819 | 0.1695 | 0.9876 |
| beagle_174.jpg | 0.8932 | 0.8979 | 0.8844 |
| beagle_183.jpg | 0.6144 | 0.6472 | 0.6577 |
| beagle_192.jpg | 0.9538 | 0.9406 | 0.9626 |
| beagle_200.jpg | 0.8273 | 0.8339 | 0.9456 |
| beagle_26.jpg | 0.5478 | 0.5788 | 0.9189 |
| beagle_35.jpg | 0.3986 | 0.2924 | 0.8946 |
| beagle_44.jpg | 0.7162 | 0.6978 | 0.9645 |
| beagle_53.jpg | 0.8831 | 0.8321 | 0.9737 |
| beagle_62.jpg | 0.8751 | 0.8573 | 0.8522 |
| beagle_71.jpg | 0.9200 | 0.8256 | 0.6172 |
| beagle_80.jpg | 0.1708 | 0.1233 | 0.9220 |
| beagle_9.jpg | 0.6841 | 0.6508 | 0.6874 |
| beagle_99.jpg | 0.8037 | 0.7469 | 0.9668 |

## Pairwise Win Rate

| Method | IG | Cheap-IG | NAA |
| --- | ---: | ---: | ---: |
| IG | — | 0.6800 | 0.2300 |
| Cheap-IG | 0.3100 | — | 0.1900 |
| NAA | 0.7700 | 0.8100 | — |

## Быстрый Доступ к Агрегатам

In [ ]:
results["summary"]


{'task': 'classifier',
 'layer_name': 'model.6',
 'n_steps': 128,
 'unit_mode': 'spatial_cell',
 'candidate_top_k': 10,
 'limit_mode': 'beam',
 'beam_size': 5,
 'n_images': 100,
 'cache_root': 'output/naopc_cache',
 'runner_kwargs': {'top_n': 0, 'fd_eps': 0.001, 'clear_every': 8},
 'method_summaries': {'IG': {'id': 'IG_84a8873b2b',
   'kind': 'ig',
   'naopc': {'n_ok': 100,
    'n_total': 100,
    'mean': 0.7286889862035423,
    'std': 0.19610162301757206,
    'median': 0.7588362299544716,
    'min': 0.11037328707855215,
    'max': 0.9920407710412111},
   'naopc_clipped': {'n_ok': 100,
    'n_total': 100,
    'mean': 0.7286889862035423,
    'std': 0.19610162301757206,
    'median': 0.7588362299544716,
    'min': 0.11037328707855215,
    'max': 0.9920407710412111},
   'aopc': {'n_ok': 100,
    'n_total': 100,
    'mean': 0.8239276962280273,
    'std': 1.5068214840990595,
    'median': 0.6255761623382569,
    'min': -3.1794761657714843,
    'max': 7.604112195968628},
   'abs_error': {'n_